# **ResNet-18 Robust**

This notebook trains a pretrained ResNet-18 on a robust dataset which combrises of a mix of SID and CIFAKE.

- Label `0`: real
- Label `1`: AI-generated/fake
- Input size: 224 x 224
- Model selection: highest validation F1

## Check that GPU exists

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available")

CUDA available: False
No GPU available


## **Mount Google Drive**

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## **Set up project**

In [3]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path("/content/ai-image-detector")

if not PROJECT_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mikkichan22/AI-image-detector.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

(PROJECT_ROOT / "src" / "__init__.py").touch()

print("Working directory:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Working directory: /content/ai-image-detector
Project exists: True
src exists: True


In [ ]:
!pip install -q datasets

## **Create sid subset**
Download 20000 SID images from the dataset

In [ ]:
!python -m src.create_sid_subset

Loading SID_Set in streaming mode...
README.md: 100% 3.30k/3.30k [00:00<00:00, 7.31MB/s]
Resolving data files: 100% 249/249 [00:00<00:00, 352950.89it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 191675.18it/s]
Resolving data files: 100% 249/249 [00:00<00:00, 221642.97it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 187639.92it/s]
Saved: {1: 504, 0: 496}
Saved: {1: 995, 0: 1005}
Saved: {1: 1487, 0: 1513}
Saved: {1: 1967, 0: 2033}
Saved: {1: 2472, 0: 2528}
Saved: {1: 2982, 0: 3018}
Saved: {1: 3494, 0: 3506}
Saved: {1: 4007, 0: 3993}
Saved: {1: 4526, 0: 4474}
Saved: {1: 5029, 0: 4971}
Saved: {1: 5543, 0: 5457}
Saved: {1: 6033, 0: 5967}
Saved: {1: 6537, 0: 6463}
Saved: {1: 7033, 0: 6967}
Saved: {1: 7556, 0: 7444}
Saved: {1: 8019, 0: 7981}
Saved: {1: 8513, 0: 8487}
Saved: {1: 9009, 0: 8991}
Saved: {1: 9527, 0: 9473}
Saved: {1: 10000, 0: 10000}

Finished.
Images saved: {1: 10000, 0: 10000}
Output: /content/ai-image-detector/data/raw/SID_Set_subset


### Check that it exists

In [ ]:
!find data/raw/SID_Set_subset -type f | wc -l
!du -sh data/raw/SID_Set_subset

20000
4.5G	data/raw/SID_Set_subset


## **Create SID splits**
Split SID images into training, validation and test splits

In [ ]:
!python -m src.create_sid_splits

Images found: 20000
Duplicate groups: 2
Created: /content/ai-image-detector/data/sid_splits.csv
Total images: 20000

Final split counts:
train      label=0: 7000
train      label=1: 7001
validation label=0: 1500
validation label=1: 1499
test       label=0: 1500
test       label=1: 1500


## **Download CIFAKE dataset**

### Upload kaggle.json file
Go to kaggle account -> settings -> API tokens -> create legacy API key

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


### Configure kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

### Download and Extract CIFAKE dataset

Download

In [ ]:
!mkdir -p /content/cifake_download
!kaggle datasets download \
    -d birdy654/cifake-real-and-ai-generated-synthetic-images \
    -p /content/cifake_download

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:01<00:00, 101MB/s]



Extract

In [4]:
!mkdir -p /content/ai-image-detector/data/raw
!unzip -q /content/cifake_download/*.zip \
    -d /content/ai-image-detector/data/raw/CIFAKE

unzip:  cannot find or open /content/cifake_download/*.zip, /content/cifake_download/*.zip.zip or /content/cifake_download/*.zip.ZIP.

No zipfiles found.


inspect the extracted structure

In [ ]:
!find /content/ai-image-detector/data/raw/CIFAKE -maxdepth 4 -type d | sort

/content/ai-image-detector/data/raw/CIFAKE
/content/ai-image-detector/data/raw/CIFAKE/test
/content/ai-image-detector/data/raw/CIFAKE/test/FAKE
/content/ai-image-detector/data/raw/CIFAKE/test/REAL
/content/ai-image-detector/data/raw/CIFAKE/train
/content/ai-image-detector/data/raw/CIFAKE/train/FAKE
/content/ai-image-detector/data/raw/CIFAKE/train/REAL


## **Create combined splits (SID and CIFAKE)**

In [ ]:
!python -m src.combine_splits

/content/ai-image-detector/src/combine_splits.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
Created: /content/ai-image-detector/data/combined_splits.csv
source_dataset  split       label
CIFAKE          test        0        10000
                            1        10378
                train       0         2500
                            1         2500
                validation  0         7500
                            1         7456
SID_Set         test        0         1500
                            1         1500
                train       0         7000
                            1         7001
                validation  0         1500
                            1 

## **Rewrite combined splits so validation set is mostly SID with some CIFAKE (60/40 ratio)**

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/ai-image-detector")

combined_path = (
    PROJECT_ROOT / "data" / "combined_splits.csv"
)

priority_path = (
    PROJECT_ROOT / "data" / "sid_priority_splits.csv"
)

combined = pd.read_csv(combined_path)

# Keep every training and test row.
non_validation = combined[
    combined["split"] != "validation"
].copy()

# Keep all SID validation rows.
sid_validation = combined[
    (combined["split"] == "validation")
    & (combined["source_dataset"] == "SID_Set")
].copy()

# Get CIFAKE validation rows.
cifake_validation = combined[
    (combined["split"] == "validation")
    & (combined["source_dataset"] == "CIFAKE")
].copy()

# Select a balanced CIFAKE validation sample.
# This selects 1,000 REAL and 1,000 FAKE images.
cifake_validation_sample = pd.concat(
    [
        group.sample(
            n=min(len(group), 1000),
            random_state=42,
        )
        for _, group in cifake_validation.groupby("label")
    ],
    ignore_index=True,
)

# Construct the new manifest.
sid_priority = pd.concat(
    [
        non_validation,
        sid_validation,
        cifake_validation_sample,
    ],
    ignore_index=True,
)

sid_priority.to_csv(
    priority_path,
    index=False,
)

print("Created:", priority_path)

Created: /content/ai-image-detector/data/sid_priority_splits.csv


### inspect result

In [ ]:
import pandas as pd

combined = pd.read_csv("data/sid_priority_splits.csv")

display(
    combined.groupby(
        ["source_dataset", "split", "label"]
    ).size()
)

source_dataset  split       label
CIFAKE          test        0        10000
                            1        10378
                train       0         2500
                            1         2500
                validation  0         1000
                            1         1000
SID_Set         test        0         1500
                            1         1499
                train       0         7000
                            1         7002
                validation  0         1500
                            1         1499
dtype: int64

### check if there's missing paths

In [ ]:
missing = []

for path_string in sid_priority["image_path"]:
    path = Path(path_string)

    if not path.is_absolute():
        path = PROJECT_ROOT / path

    if not path.exists():
        missing.append(str(path))

print("Missing paths:", len(missing))

Missing paths: 0


# **Train one epoch as a test**

In [ ]:
!python -m src.train_resnet \
    --project_root . \
    --splits_file data/combined_splits.csv \
    --output_dir /content/drive/MyDrive/ai-image-detector/results/sid_mixed_test \
    --checkpoint_dir /content/drive/MyDrive/ai-image-detector/checkpoints/sid_mixed_test \
    --epochs 1 \
    --freeze_epochs 1 \
    --batch_size 32 \
    --num_workers 2 \
    --seed 42

Using device: cuda
Reading split file: data/combined_splits.csv
Training images: 22001
Validation images: 17955
Test images: 20378
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 201MB/s]

Epoch 1/1
Train loss: 0.5806, Train F1: 0.6986
Validation loss: 0.5549, Validation F1: 0.7598
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_mixed_test/resnet18_clean_best.pth
                                     
Final test results:
Accuracy:  0.7093
Precision: 0.6600
Recall:    0.8855
F1:        0.7563

Saved:
/content/drive/MyDrive/ai-image-detector/results/sid_mixed_test/training_history.csv
/content/drive/MyDrive/ai-image-detector/results/sid_mixed_test/test_metrics.json


# **Train final model**

In [ ]:
!python -m src.train_resnet \
    --project_root . \
    --splits_file data/sid_priority_splits.csv \
    --output_dir /content/drive/MyDrive/ai-image-detector/results/sid_priority_augmented \
    --checkpoint_dir /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented \
    --epochs 10 \
    --freeze_epochs 2 \
    --batch_size 64 \
    --num_workers 2 \
    --seed 42

Using device: cuda
Reading split file: data/sid_priority_splits.csv
Training images: 19002
Validation images: 4999
Test images: 23377
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 173MB/s]

Epoch 1/10
Train loss: 0.6252, Train F1: 0.6590
Validation loss: 0.5581, Validation F1: 0.7581
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented/resnet18_clean_best.pth

Epoch 2/10
Train loss: 0.5128, Train F1: 0.7681
Validation loss: 0.4831, Validation F1: 0.8101
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented/resnet18_clean_best.pth

Epoch 3/10
Unfreezing full ResNet-18
Train loss: 0.2801, Train F1: 0.8914
Validation loss: 0.1774, Validation F1: 0.9323
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented/resnet18_clean_best.pth

Epoch 4/10
Train loss: 0.1471, Train F1: 0.9448
Validatio

# **Model evaluation**

In [ ]:
import json

metrics_path = (
    "/content/drive/MyDrive/ai-image-detector/"
    "results/sid_priority_augmented/test_metrics.json"
)

with open(metrics_path) as file:
    metrics = json.load(file)

print(json.dumps(metrics, indent=2))

{
  "model": "resnet18",
  "training_type": "clean",
  "best_epoch": 10,
  "best_validation_f1": 0.9714173073132268,
  "test_accuracy": 0.9445181160970184,
  "test_precision": 0.9263378465506125,
  "test_recall": 0.9677527995285005,
  "test_f1": 0.9465925468396129,
  "confusion_matrix": [
    [
      10586,
      914
    ],
    [
      383,
      11494
    ]
  ],
  "classification_report": {
    "real": {
      "precision": 0.9650834169021789,
      "recall": 0.9205217391304348,
      "f1-score": 0.9422760247452046,
      "support": 11500.0
    },
    "AI/fake": {
      "precision": 0.9263378465506125,
      "recall": 0.9677527995285005,
      "f1-score": 0.9465925468396129,
      "support": 11877.0
    },
    "accuracy": 0.9445181160970184,
    "macro avg": {
      "precision": 0.9457106317263957,
      "recall": 0.9441372693294676,
      "f1-score": 0.9444342857924087,
      "support": 23377.0
    },
    "weighted avg": {
      "precision": 0.9453982075483031,
      "recall": 0.94451

## **Overall Performance**

| Metric         | Result | Meaning                                            |
| -------------- | -----: | -------------------------------------------------- |
| Test accuracy  | 94.45% | 94.45% of test images were classified correctly    |
| Test precision | 92.63% | Of images predicted as AI, 92.63% were actually AI |
| Test recall    | 96.78% | The model detected 96.78% of AI images             |
| Test F1        | 94.66% | Overall balance between precision and recall       |


**Real images:**
Correctly classified as real: 10,586
Incorrectly classified as AI: 914

Approximately 8% of real images are false positives:


**AI images:**
Correctly classified as AI: 11,494
Incorrectly classified as real: 383

Only approximately 3.2% of AI images were missed.

**What this says about the model:**

The model is optimized toward detecting AI images aggressively:

## **Load the trained model**

In [ ]:
import os
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path("/content/ai-image-detector")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train_resnet import build_model, ManifestDataset
from src.transforms import evaluation_transform

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = Path(
    "/content/drive/MyDrive/ai-image-detector/"
    "checkpoints/sid_priority_augmented/"
    "resnet18_clean_best.pth"
)

print("Checkpoint exists:", checkpoint_path.exists())

model = build_model()

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Model loaded successfully")
print("Using device:", device)

Checkpoint exists: True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s]


Model loaded successfully
Using device: cpu


## **Evaluate for CIFAKE and SID seperately**
Since testing split for CIFAKE is a lot larger than SID, we test them separately to see how the model performs on the 2 datasets

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

splits = pd.read_csv(
    "data/sid_priority_splits.csv"
)

test_rows = splits[
    splits["split"] == "test"
]

for source_name in ["SID_Set", "CIFAKE"]:

    source_rows = test_rows[
        test_rows["source_dataset"] == source_name
    ].reset_index(drop=True)

    dataset = ManifestDataset(
        source_rows,
        PROJECT_ROOT,
        evaluation_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels_batch in loader:
            images = images.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels_batch.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    print(f"\n===== {source_name} TEST RESULTS =====")
    print("Images:", len(source_rows))

    print(
        "Accuracy:",
        accuracy_score(
            all_labels,
            all_predictions,
        ),
    )

    print(
        "Precision:",
        precision_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        "Recall:",
        recall_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        "F1:",
        f1_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        classification_report(
            all_labels,
            all_predictions,
            target_names=["REAL", "AI/fake"],
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            all_labels,
            all_predictions,
            labels=[0, 1],
        )
    )


===== SID_Set TEST RESULTS =====
Images: 3000
Accuracy: 0.993
Precision: 0.9881188118811881
Recall: 0.998
F1: 0.9930348258706467
              precision    recall  f1-score   support

        REAL       1.00      0.99      0.99      1500
     AI/fake       0.99      1.00      0.99      1500

    accuracy                           0.99      3000
   macro avg       0.99      0.99      0.99      3000
weighted avg       0.99      0.99      0.99      3000

Confusion matrix:
[[1482   18]
 [   3 1497]]

===== CIFAKE TEST RESULTS =====
Images: 20378
Accuracy: 0.9384139758563156
Precision: 0.9192940527622024
Recall: 0.9636731547504336
F1: 0.9409606247353813
              precision    recall  f1-score   support

        REAL       0.96      0.91      0.94     10000
     AI/fake       0.92      0.96      0.94     10378

    accuracy                           0.94     20378
   macro avg       0.94      0.94      0.94     20378
weighted avg       0.94      0.94      0.94     20378

Confusion matri